# ASRS LoRA Classifier

### First look at the data - loading one file and looking at it

In [28]:
import pandas as pd

DATA_FOLDER = "data/raw/nasa/"
path = "data/raw/nasa/asrs_2010_H1.csv"
raw = pd.read_csv(path, header=None, dtype=str, nrows=5)
print(raw.shape)
raw.iloc[:, :6]

(5, 126)


,0,1,2,3,4,5
0,,Time,Time,Place,Place,Place
1,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial
2,866849,201001,1201-1800,ZHU.ARTCC,TX,NaN
3,866859,201001,1201-1800,ZAU.ARTCC,IL,NaN
4,866866,201001,1801-2400,ZHU.ARTCC,TX,NaN


### We build the column names from the groups (row 0) and the fields (row 1)
By taking row 0 and 1, pairing them column by column and transforming each pair to a column name of the form {group} | {field}

Example:
- Row 0, Col 2 = Time
- Row 1, Col 2 = Local Time Of Day
- Therefore we get the new column name: "Time | Local Time Of Day"


In [ ]:
print(repr(raw.iloc[0, 0]))

groups = raw.iloc[0].str.strip()
fields = raw.iloc[1].str.strip()

names = []
for g, f in zip(groups, fields):
    if g == "":
        names.append(f)
    else:
        names.append(f"{g} | {f}")

names[:6]

' '


['ACN',
 'Time | Date',
 'Time | Local Time Of Day',
 'Place | Locale Reference',
 'Place | State Reference',
 'Place | Relative Position.Angle.Radial']

In [25]:
len(names) == len(set(names))

True

In [29]:
[n for n in names if "Anomaly" in n or "Narrative" in n]

['Events | Anomaly', 'Report 1 | Narrative', 'Report 2 | Narrative']

### Load the full file, real column names, only what we need

In [33]:
df = pd.read_csv(path, header=None, skiprows=2, names=names, dtype=str)
df.shape

(2814, 126)

In [34]:
cols = {
    "ACN": "acn",
    "Time | Date": "date",
    "Events | Anomaly": "anomaly",
    "Report 1 | Narrative": "narrative",
}
df = df[list(cols)].rename(columns=cols)
df.head()

,acn,date,anomaly,narrative
0,866849,201001,ATC Issue All Types; Deviation / Discrepancy -...,Ocean West and Offshore West/Central were comb...
1,866859,201001,ATC Issue All Types; Deviation / Discrepancy -...,Aircraft X had a flight plan in URET (user req...
2,866866,201001,ATC Issue All Types; Airspace Violation All Ty...,I was working D-79 when Aircraft X reported ov...
3,866977,201001,Aircraft Equipment Problem Critical,Descending to FL340 we got a Green Hydraulic L...
4,866984,201001,Aircraft Equipment Problem Critical; Inflight ...,On climbout at approximately 15;000 FT noticed...


In [35]:
df.isna().sum()

acn           0
date          0
anomaly      12
narrative     0
dtype: int64

We can see that we get 12 missing anomalies. These rows will be dropped in the next step.

### From the anomaly string to a list of labels

In [ ]:
# Drop the 12 reports with no anomaly
df = df.dropna(subset=["anomaly"])
df.shape

(2802, 4)

In [42]:
len(set(df["anomaly"]))

685

In [41]:
ORPHANS = {"Critical", "Less Severe"}

def parse_anomaly(s):
    labels = []
    for part in s.split(";"):
        part = part.strip()
        if part == "":
            continue
        if part in ORPHANS:
            assert labels[-1] == "Conflict Ground Conflict", (s, part)
            labels[-1] = f"{labels[-1]} {part}"
        else:
            labels.append(part)
    return labels

In [43]:
parse_anomaly("ATC Issue All Types; Conflict Ground Conflict; Critical; Ground Incursion Runway")

['ATC Issue All Types',
 'Conflict Ground Conflict Critical',
 'Ground Incursion Runway']

In [44]:
df["labels"] = df["anomaly"].apply(parse_anomaly)
df["labels"].explode().value_counts()

labels
Deviation / Discrepancy - Procedural Published Material / Policy     1014
Aircraft Equipment Problem Critical                                   868
ATC Issue All Types                                                   680
Aircraft Equipment Problem Less Severe                                435
Deviation / Discrepancy - Procedural Clearance                        386
Deviation - Track / Heading All Types                                 213
Conflict Airborne Conflict                                            212
Inflight Event / Encounter Weather / Turbulence                       156
Deviation / Discrepancy - Procedural FAR                              140
Deviation / Discrepancy - Procedural Maintenance                      112
Deviation - Altitude Excursion From Assigned Altitude                 107
Airspace Violation All Types                                           93
Conflict NMAC                                                          93
Inflight Event / Encounter Loss